In [ ]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('Realestate.csv')
df

,No,X1 transaction date,X2 house age,X3 distance to the nearest MRT station,X4 number of convenience stores,X5 latitude,X6 longitude,Y house price of unit area
0,1,2012.917,32.0,84.87882,10,24.98298,121.54024,37.9
1,2,2012.917,19.5,306.59470,9,24.98034,121.53951,42.2
2,3,2013.583,13.3,561.98450,5,24.98746,121.54391,47.3
3,4,2013.500,13.3,561.98450,5,24.98746,121.54391,54.8
4,5,2012.833,5.0,390.56840,5,24.97937,121.54245,43.1
...,...,...,...,...,...,...,...,...
409,410,2013.000,13.7,4082.01500,0,24.94155,121.50381,15.4
410,411,2012.667,5.6,90.45606,9,24.97433,121.54310,50.0
411,412,2013.250,18.8,390.96960,7,24.97923,121.53986,40.6
412,413,2013.000,8.1,104.81010,5,24.96674,121.54067,52.5


In [ ]:
df['X1 transaction date']=df['X1 transaction date'].astype(int)

In [ ]:
new_names = ['No','date','age','dist','store','lat','long','price']
df = df.set_axis(new_names, axis="columns")

In [ ]:
df.set_index('No')

,date,age,dist,store,lat,long,price
No,,,,,,,
1,2012,32.0,84.87882,10,24.98298,121.54024,37.9
2,2012,19.5,306.59470,9,24.98034,121.53951,42.2
3,2013,13.3,561.98450,5,24.98746,121.54391,47.3
4,2013,13.3,561.98450,5,24.98746,121.54391,54.8
5,2012,5.0,390.56840,5,24.97937,121.54245,43.1
...,...,...,...,...,...,...,...
410,2013,13.7,4082.01500,0,24.94155,121.50381,15.4
411,2012,5.6,90.45606,9,24.97433,121.54310,50.0
412,2013,18.8,390.96960,7,24.97923,121.53986,40.6


In [ ]:
print(df.shape)

(414, 8)


In [ ]:
Q1 = df.quantile(0.25)
Q3 = df.quantile(0.75)
IQR = Q3 - Q1
df = df[~((df < (Q1 - 1.5 * IQR)) | (df > (Q3 + 1.5 * IQR))).any(axis=1)]
print(df.shape)

(371, 8)


In [ ]:
pip install geopy

In [ ]:
pip install googletrans==4.0.0-rc1

In [ ]:
from geopy.geocoders import Nominatim
from googletrans import Translator
import time
import pandas as pd

# Menambahkan delay selama 1 detik setelah setiap permintaan
time.sleep(1)

# Inisialisasi objek geocoder
geolocator = Nominatim(user_agent="my_geocoder")

# Inisialisasi objek terjemahan
translator = Translator()

# Fungsi untuk mendapatkan nama desa yang diterjemahkan
def get_translated_village_name(row):
    location = geolocator.reverse((row['lat'], row['long']), exactly_one=True)

    # Mendapatkan alamat lengkap dalam bahasa lokal dari data geokoding
    alamat_lokal = location.raw.get('address', {})

    # Pilih nama desa berdasarkan beberapa kunci yang mungkin ada
    nama_desa_lokal = alamat_lokal.get('village') or alamat_lokal.get('suburb') or alamat_lokal.get('city') or ''

    # Menerjemahkan nama desa ke bahasa Inggris
    try:
        terjemahan = translator.translate(nama_desa_lokal, src='auto', dest='en')
        return terjemahan.text
    except Exception as e:
        return ''

# Membuat salinan df ke df_6
df_6 = df.copy()

# Menambahkan kolom 'nama_desa_inggris' ke df_6
df_6['nama_desa_inggris'] = df_6.apply(get_translated_village_name, axis=1)

# Menampilkan DataFrame df_6 dengan kolom 'nama_desa_inggris' yang baru
print(df_6)


In [ ]:
unique_distrik = df_6['nama_desa_inggris'].unique()
print(unique_distrik)

In [ ]:
df_6

In [ ]:
frekuensi_distrik = df_6['nama_desa_inggris'].value_counts()
print(frekuensi_distrik)

Dapinglin     258
New store     101
Pond            8
Fulfilling      4
Name: nama_desa_inggris, dtype: int64


In [ ]:
import folium

# Inisialisasi peta dengan koordinat tengah (misalnya, rata-rata latitude dan longitude)
LAT = df_6['lat']
LONG = df_6['long']
CPRICE = df_6['price']
peta = folium.Map(location=[LAT.mean(), LONG.mean()], zoom_start=15)

# Tentukan rentang warna yang akan digunakan (misalnya, dari hijau ke merah)
colors = ['green', 'green', 'yellow', 'red']

# Tentukan rentang nilai untuk kolom house_price (anda bisa sesuaikan sesuai data yang sesungguhnya)
price_ranges = [0, 20, 40, 50, float('inf')]

# Tambahkan penanda untuk setiap titik data dengan warna yang sesuai
for i in range(len(LAT)):
    for j in range(len(price_ranges) - 1):
        if price_ranges[j] <= df_6['price'].iloc[i] < price_ranges[j+1]:
            folium.CircleMarker(
                location=[LAT.iloc[i], LONG.iloc[i]],
                radius=5,  # Ukuran marker
                color=colors[j],  # Warna marker
                fill=True,
                fill_color=colors[j],
                fill_opacity=0.6,
                popup=f'Harga Rumah: {df_6["price"].iloc[i]:.2f}'  # Popup informasi
            ).add_to(peta)

# Tampilkan peta
peta

In [ ]:
df_6.rename(columns={'nama_desa_inggris': 'village'}, inplace=True)

In [ ]:
df_6

,No,date,age,dist,store,lat,long,price,Village
0,1,2012,32.0,84.87882,10,24.98298,121.54024,37.9,Dapinglin
1,2,2012,19.5,306.59470,9,24.98034,121.53951,42.2,Dapinglin
2,3,2013,13.3,561.98450,5,24.98746,121.54391,47.3,Dapinglin
3,4,2013,13.3,561.98450,5,24.98746,121.54391,54.8,Dapinglin
4,5,2012,5.0,390.56840,5,24.97937,121.54245,43.1,Dapinglin
...,...,...,...,...,...,...,...,...,...
408,409,2013,18.5,2175.74400,3,24.96330,121.51243,28.1,New store
410,411,2012,5.6,90.45606,9,24.97433,121.54310,50.0,Dapinglin
411,412,2013,18.8,390.96960,7,24.97923,121.53986,40.6,Dapinglin
412,413,2013,8.1,104.81010,5,24.96674,121.54067,52.5,Dapinglin


In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Ganti path dengan direktori yang Anda inginkan di Google Drive
directory = "/content/drive/MyDrive/Model"

# Buat direktori jika belum ada
os.makedirs(directory, exist_ok=True)

# Simpan DataFrame df_6 ke dalam file CSV di direktori yang ditentukan
file_path = os.path.join(directory, "Realestate_edit.csv")
df_6.to_csv(file_path, index=False)

Mounted at /content/drive
